In [ ]:
%pip install -q vl-convert-python

import os
import altair as alt
import pandas as pd
from PIL import Image

alt.data_transformers.disable_max_rows()
print("Altair versión:", alt.__version__)

Altair versión: 5.5.0


In [ ]:
from google.colab import files

subido = files.upload()
nombre = list(subido.keys())[0]

datos = pd.read_csv(nombre, sep=";", encoding="utf-8")
print("Archivo:", nombre)
print("Personas (filas):", len(datos))
print("Columnas:", list(datos.columns))

Saving killed_in_gaza_limpiado_3.csv to killed_in_gaza_limpiado_3.csv
Archivo: killed_in_gaza_limpiado_3.csv
Personas (filas): 13070
Columnas: ['id', 'in_name', 'ar_name', 'edad', 'dob', 'gen', 'act', 'grupo']


In [ ]:
datos["grupo"] = datos["grupo"].astype(str).str.strip().str.upper()
datos["act"] = pd.to_numeric(datos["act"]).astype(int)

columnas_personales = [c for c in ["id", "in_name", "ar_name", "dob"] if c in datos.columns]
datos = datos.drop(columns=columnas_personales)

print("Columnas eliminadas:", columnas_personales)
print("Columnas que quedan:", list(datos.columns))
print()
print(datos["grupo"].value_counts())

Columnas eliminadas: ['id', 'in_name', 'ar_name', 'dob']
Columnas que quedan: ['edad', 'gen', 'act', 'grupo']

grupo
MADRE POTENCIAL    10946
BEBE                2124
Name: count, dtype: int64


In [ ]:
import numpy as np
import pandas as pd

# --- 1) Los períodos: qué actualizaciones (act) se unen en cada uno
periodo_de_act = {1: 1, 2: 2, 3: 2, 4: 2, 5: 2, 6: 3, 7: 4, 8: 4, 9: 4, 10: 4}

nombres_periodo = {
    1: "Noviembre 2023: Inicio de la ofensiva israelí y primeros bombardeos aéreos",
    2: "Agosto 2024: Expansión terrestre y ofensiva sobre Rafah",
    3: "Marzo 2025: Tregua de Enero, remoción de escombros y ruptura",
    4: "Mayo 2026: Reanudación de hostilidades",
}

# Cantidades que esperas por período (sirven para comprobar que todo calza)
esperado = {1: 525, 2: 583, 3: 608, 4: 408}

# --- 2) Solo los bebés, con su período, ordenados del más antiguo al más reciente
bebes = datos[datos["grupo"] == "BEBE"].copy()
bebes["periodo"] = bebes["act"].map(periodo_de_act)

if bebes["periodo"].isna().any():
    raise ValueError("Hay actualizaciones (act) que no están asignadas a ningún período.")

bebes["periodo"] = bebes["periodo"].astype(int)
bebes = bebes.sort_values(["periodo", "act"]).reset_index(drop=True)
N = len(bebes)

# --- 3) Comparar con las cantidades esperadas
conteo = bebes.groupby("periodo").size()
for p in sorted(esperado):
    real = int(conteo.get(p, 0))
    marca = "OK" if real == esperado[p] else "ATENCIÓN: no coincide"
    print(f"Período {p}: {real} bebés (esperado {esperado[p]}) -> {marca}")

etiquetas = dict(nombres_periodo)
orden_etiquetas = [etiquetas[p] for p in sorted(etiquetas)]


# --- 4) La silueta: dice si un punto (x, y) cae dentro de la figura
def elipse(x, y, cx, cy, rx, ry):
    return ((x - cx) / rx) ** 2 + ((y - cy) / ry) ** 2 <= 1


def dentro(x, y):
    # Proporciones de bebé: cabeza muy grande, cuerpo redondo, brazos y piernas cortos
    cabeza = elipse(x, y, 0, 6.7, 2.5, 2.5)
    cuerpo = elipse(x, y, 0, 3.0, 1.9, 1.9)
    brazo_i = elipse(x, y, -2.7, 3.3, 0.7, 1.3)
    brazo_d = elipse(x, y, 2.7, 3.3, 0.7, 1.3)
    pierna_i = elipse(x, y, -0.9, 0.9, 0.8, 0.9)
    pierna_d = elipse(x, y, 0.9, 0.9, 0.8, 0.9)
    return cabeza | cuerpo | brazo_i | brazo_d | pierna_i | pierna_d


def grilla(escala):
    limite_x = int(3.8 * escala) + 2
    limite_y = int(9.8 * escala) + 2
    gx, gy = np.meshgrid(np.arange(-limite_x, limite_x + 1) + 0.5,
                         np.arange(0, limite_y + 1) + 0.5)
    gx, gy = gx.ravel(), gy.ravel()
    mascara = dentro(gx / escala, gy / escala)
    return gx[mascara], gy[mascara]


# --- 5) Buscar el tamaño de silueta que necesita justo N puntos
bajo, alto = 1.0, 60.0
for _ in range(40):
    medio = (bajo + alto) / 2
    if len(grilla(medio)[0]) >= N:
        alto = medio
    else:
        bajo = medio

px, py = grilla(alto)

# Si sobran unos pocos puntos, se quitan al azar para que queden exactamente N
sobran = len(px) - N
if sobran > 0:
    azar = np.random.default_rng(0)
    quitar = azar.choice(len(px), sobran, replace=False)
    dejar = np.ones(len(px), dtype=bool)
    dejar[quitar] = False
    px, py = px[dejar], py[dejar]

# --- 6) Ordenar los puntos de abajo hacia arriba y repartirlos por período
orden = np.lexsort((px, py))
px, py = px[orden], py[orden]

figura = pd.DataFrame({
    "x": px,
    "y": py,
    "periodo": bebes["periodo"].values,
})
figura["etiqueta"] = figura["periodo"].map(etiquetas)
figura["total_periodo"] = figura.groupby("periodo")["periodo"].transform("count")

print()
print("Bebés:", N, "| Puntos en la silueta:", len(figura))
figura.head()

Período 1: 525 bebés (esperado 525) -> OK
Período 2: 583 bebés (esperado 583) -> OK
Período 3: 608 bebés (esperado 608) -> OK
Período 4: 408 bebés (esperado 408) -> OK

Bebés: 2124 | Puntos en la silueta: 2124


,x,y,periodo,etiqueta,total_periodo
0,-8.5,0.5,1,Noviembre 2023: Inicio de la ofensiva israelí ...,525
1,-7.5,0.5,1,Noviembre 2023: Inicio de la ofensiva israelí ...,525
2,-6.5,0.5,1,Noviembre 2023: Inicio de la ofensiva israelí ...,525
3,-5.5,0.5,1,Noviembre 2023: Inicio de la ofensiva israelí ...,525
4,-4.5,0.5,1,Noviembre 2023: Inicio de la ofensiva israelí ...,525


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, to_hex

PX = 7

# Degradado: gris -> crema -> rojo sangre, un color por cada período
degradado = LinearSegmentedColormap.from_list(
    "gris_crema_rojo", ["#AAA6A1", "#F1EDE6", "#9E1724"]
)
colores = [
    to_hex(degradado(i / (len(orden_etiquetas) - 1)))
    for i in range(len(orden_etiquetas))
]

total_txt = f"{N:,}".replace(",", ".")

x_min, x_max = figura["x"].min() - 1, figura["x"].max() + 1
y_min, y_max = figura["y"].min() - 1, figura["y"].max() + 1

grafico = alt.Chart(figura[["x", "y", "periodo", "etiqueta", "total_periodo"]]).mark_circle(size=22, opacity=1).encode(
    x=alt.X("x:Q", axis=None, scale=alt.Scale(domain=[x_min, x_max], nice=False)),
    y=alt.Y("y:Q", axis=None, scale=alt.Scale(domain=[y_min, y_max], nice=False)),
    color=alt.Color("etiqueta:N",
                    sort=orden_etiquetas,
                    scale=alt.Scale(domain=orden_etiquetas, range=colores),
                    legend=alt.Legend(title="Período",
                                      symbolType="circle", symbolSize=140,
                                      labelLimit=700)),
    tooltip=[alt.Tooltip("etiqueta:N", title="Período"),
             alt.Tooltip("total_periodo:Q", title="Bebés registrados en el período")]
).properties(
    width=int((x_max - x_min) * PX),
    height=int((y_max - y_min) * PX),
    title=alt.TitleParams(
        text=["Cada punto representa a un bebé", f"de 0 a 1 año: {total_txt} en total"],
        subtitle=["Los colores indican el período de publicación de la lista",
                  "en que fue registrado: del más antiguo (abajo)",
                  "al más reciente (arriba)."],
        color="white", subtitleColor="#cccccc", anchor="start"
    )
).interactive(
).configure(
    background="#111111"
).configure_view(
    stroke=None
).configure_legend(
    labelColor="white", titleColor="white"
)

grafico

alt.Chart(...)

In [ ]:
assert len(figura) == len(bebes), "Los puntos no coinciden con los bebés"

conteo_real = bebes.groupby("periodo").size()
conteo_figura = figura.groupby("periodo").size()
assert (conteo_real == conteo_figura).all(), "Algún total por período no coincide"

for p, esp in esperado.items():
    real = int(conteo_real.get(p, 0))
    assert real == esp, f"Período {p}: la tabla tiene {real} bebés y esperabas {esp}"

print("Verificación aprobada.")
print("Total de bebés en el gráfico:", len(figura))

Verificación aprobada.
Total de bebés en el gráfico: 2124


In [ ]:
CARPETA_SALIDA = "salida"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

ruta_html = os.path.join(CARPETA_SALIDA, "grafico.html")
ruta_png = os.path.join(CARPETA_SALIDA, "grafico.png")
ruta_jpg = os.path.join(CARPETA_SALIDA, "grafico.jpg")

grafico.save(ruta_html)
grafico.save(ruta_png, scale_factor=2)
Image.open(ruta_png).convert("RGB").save(ruta_jpg, quality=95)

print("Archivos guardados:", sorted(os.listdir(CARPETA_SALIDA)))

from google.colab import files
files.download(ruta_html)
files.download(ruta_jpg)

Archivos guardados: ['grafico.html', 'grafico.jpg', 'grafico.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>